[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/experiment/outline-and-fragment-fixes-2026-10-28/notebooks/08_tone_matched_rerun.ipynb)

# 08 — Do brighter 2023 photos find the houses the detector missed?

**Question.** On the Eastern Shore the detector finds 54% of hand-traced houses on the 2023-06-02 NAIP flight and 66% on 2023-08-19, but 83% on 2023-05-27. The 2023 photos are much darker than 2021 (RGB mean ~71 vs ~100 at the same places). If each 2023 tile is re-toned to the 2021 photo of the same square before detection, how many more traced houses are found, and how many more false detections appear?

**How.** Same tiles, same 2023 images (the NAIP item IDs recorded by notebook 05), same detector and filter. The only change: each band's histogram is matched to the 2021 photo of the same 2 km square (`tiles.fetch_tile_matched`). Tiles: every Eastern Shore tile from those two flights that contains at least one USGS-traced house (chosen by the tracing, not by what the detector found, so the comparison is fair).

**Checked against.** USGS hand tracing (2016/17), same scoring as notebook 05, before vs after on exactly the same tiles. A small test on 26 missed houses recovered 6 and lost none of 11 found ones.

**Outputs.** `Task1_results/experiments/tone_match_2021/` (per-county run folders, `comparison.csv`). Nothing in `county_*` is changed.
**Scale.** `RESOLUTION_M` sets the pixel size the 2023 photo is read at (default 1.0 m, as in notebook 05). A 26-house test found 0.8 m + tone matching recovers about twice as many missed houses as 1.0 m + tone matching; a run with `RESOLUTION_M = 0.8` writes to its own folder (`..._res0.8m`).
**Runtime.** Two photos per tile (2023 + 2021) → roughly twice notebook 05's time per tile. Resumable: Run all again after a crash.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "experiment/outline-and-fragment-fixes-2026-10-28"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, tiles, region, viz
from geo_anom.task1 import county as C
from geo_anom.task1.kmz import write_kmz
from geo_anom.task1.pipeline import run_pipeline
from shapely.geometry import box
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
FLIGHTS = ["2023-06-02", "2023-08-19"]     # the two worst flights on the Eastern Shore
REFERENCE_YEAR = "2021"
COUNTIES = C.EASTERN_SHORE                  # or e.g. ["Worcester"] for a quicker test
RESOLUTION_M = 1.0                          # pixel size the photo is read at; 0.8 found 2x more missed houses in a 26-house test
MAX_WORKERS = 12
tiles.TARGET_RESOLUTION_M = RESOLUTION_M
EXP = env.results_root() / "experiments" / (f"tone_match_{REFERENCE_YEAR}" + ("" if RESOLUTION_M == 1.0 else f"_res{RESOLUTION_M:g}m"))
gt = load_ground_truth().to_crs(region.UTM)
print("results ->", EXP)

## 1. Which tiles

In [ ]:
def tile_box(s):
    return gpd.GeoSeries([box(*s["bbox"])], crs=4326).to_crs(region.UTM).iloc[0]

selected = {}
for n in C.EASTERN_SHORE:
    if n not in COUNTIES:
        continue
    sites = json.loads((C.county_out(n) / "sites.json").read_text())
    cu = region.county_boundary(n).to_crs(region.UTM).unary_union
    g = gt[gt.centroid.within(cu)].centroid
    sel = [s for s in sites if (s.get("naip_datetime") or "")[:10] in FLIGHTS and s.get("bbox")
           and g.within(tile_box(s)).any()]
    if sel:
        selected[n] = sel
display(pd.Series({n: len(v) for n, v in selected.items()}, name="tiles"))
print(sum(len(v) for v in selected.values()), "tiles in total")

## 2. Run (resumable)

In [ ]:
from tqdm.auto import tqdm
for n, sel in selected.items():
    out = EXP / C.county_out(n).name
    print(f"\n=== {n} ===")
    try:
        run_pipeline(sel, out, manifest=tiles.load_registry(), manifest_path=tiles.REGISTRY_SNAPSHOT,
                     prefer="area", stream=True, max_workers=MAX_WORKERS, progress=tqdm,
                     filters={"farm": [], "county": [n]}, fetch=tiles.fetch_tile_matched)
    except Exception as e:
        print(f"{n}: FAILED ({type(e).__name__}: {e}) -- run again to retry")
    finally:
        import gc, torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 3. Before vs after, same tiles

Only traced houses and detections whose centre lies in the selected tiles (and in the county) count. *Found* = traced houses touched by a detection; *precision* = detections touching a traced house (detections that touch none may still be houses built after 2016/17).

In [ ]:
def score(b, g):
    hb = gpd.sjoin(b[["geometry"]], g[["geometry"]], predicate="intersects") if len(b) and len(g) else None
    hit_b = 0 if hb is None else hb.index.nunique()
    hit_g = 0 if hb is None else hb.index_right.nunique()
    return len(b), hit_b, hit_g

rows = []
for n, sel in selected.items():
    out = EXP / C.county_out(n).name
    if not (out / "buildings.geojson").exists():
        print(n, "not run yet"); continue
    cu = region.county_boundary(n).to_crs(region.UTM).unary_union
    area = gpd.GeoSeries([tile_box(s) for s in sel], crs=region.UTM).unary_union.intersection(cu)
    g = gt[gt.centroid.within(area)]
    _, _, before = C.load_county_run(C.county_out(n))
    _, _, after = C.load_county_run(out)
    pick = lambda b: b.to_crs(region.UTM)[b.to_crs(region.UTM).centroid.within(area)] if len(b) else b
    for label, b in [("before", pick(before)), ("after (tone-matched)", pick(after))]:
        d, hb, hg = score(b, g)
        rows.append({"county": n, "run": label, "traced houses": len(g), "detections": d,
                     "traced found": hg, "recall": hg / len(g) if len(g) else None,
                     "precision": hb / d if d else None})
comp = pd.DataFrame(rows)
tot = comp.groupby("run")[["traced houses", "detections", "traced found"]].sum()
hits = {r: comp[comp.run == r].eval("precision * detections").sum() for r in tot.index}
tot["recall"] = tot["traced found"] / tot["traced houses"]
tot["precision"] = [hits[r] / tot.loc[r, "detections"] for r in tot.index]
display(comp.round(3)); display(Markdown("**All selected tiles**")); display(tot.round(3))
comp.to_csv(EXP / "comparison.csv", index=False)

## What this shows

*(Write after the run: recall and precision before vs after on the same tiles, per county; whether tone matching is worth applying to every 2023 tile.)*